# Unidad 4 · Colab 3 de 3
## Aplicación práctica: microservicio de predicciones y métricas de negocio

**Objetivos de este notebook**

- Diseñar un microservicio FastAPI que expone predicciones y métricas para que las consuma una app web.
- Manejar errores HTTP apropiados (`404`, `422`, `500`) con `HTTPException`.
- Habilitar CORS para que un frontend en otro dominio pueda consumir la API.
- Conectar esto con lo visto en las Unidades 5 y 6: cómo se consume desde otra app y cómo se despliega.

> **Nivel:** intermedio. Este notebook integra todo lo visto en los Colab 1 y 2 de esta unidad.

---

## 1. Diseño del microservicio

Vamos a construir una API que un equipo de producto podría consumir desde una app web, con dos capacidades típicas de un microservicio de datos:

- `POST /predict` — recibe features de un cliente y devuelve una predicción (por ejemplo, probabilidad de abandono/churn).
- `GET /metrics` — devuelve métricas de negocio agregadas (por ejemplo, ventas por período), con filtros por query params.
- `GET /health` — endpoint de salud, típico para que la plataforma de despliegue (Unidad 6) sepa si el servicio está vivo.

## 2. Modelos de entrada y salida para la predicción

```python
from pydantic import BaseModel, Field
from typing import Literal

class ClienteFeatures(BaseModel):
    antiguedad_meses: int = Field(ge=0)
    gasto_mensual: float = Field(ge=0)
    reclamos_ultimo_trimestre: int = Field(ge=0)

class PrediccionOut(BaseModel):
    probabilidad_churn: float
    riesgo: Literal['bajo', 'medio', 'alto']
```

Separar `ClienteFeatures` (entrada) de `PrediccionOut` (salida) deja claro qué espera recibir el servicio y qué garantiza devolver — el contrato de la API.

In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class ClienteFeatures(BaseModel):
    antiguedad_meses: int = Field(ge=0)
    gasto_mensual: float = Field(ge=0)
    reclamos_ultimo_trimestre: int = Field(ge=0)

class PrediccionOut(BaseModel):
    probabilidad_churn: float
    riesgo: Literal['bajo', 'medio', 'alto']

In [2]:
!pip install -q fastapi uvicorn

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Literal

app = FastAPI(title='Microservicio de Predicciones', version='1.0.0')

class ClienteFeatures(BaseModel):
    antiguedad_meses: int = Field(ge=0)
    gasto_mensual: float = Field(ge=0)
    reclamos_ultimo_trimestre: int = Field(ge=0)

class PrediccionOut(BaseModel):
    probabilidad_churn: float
    riesgo: Literal['bajo', 'medio', 'alto']

def predecir_churn(features: ClienteFeatures) -> PrediccionOut:
    # Heuristica simple en lugar de un modelo entrenado, solo para el ejemplo
    score = 0.1 * features.reclamos_ultimo_trimestre - 0.01 * features.antiguedad_meses
    probabilidad = max(0.0, min(1.0, 0.3 + score))
    riesgo = 'alto' if probabilidad > 0.6 else 'medio' if probabilidad > 0.3 else 'bajo'
    return PrediccionOut(probabilidad_churn=round(probabilidad, 2), riesgo=riesgo)

@app.post('/predict', response_model=PrediccionOut)
def predict(features: ClienteFeatures):
    return predecir_churn(features)

cliente = TestClient(app)
resp = cliente.post('/predict', json={'antiguedad_meses': 3, 'gasto_mensual': 50, 'reclamos_ultimo_trimestre': 4})
print(resp.status_code, resp.json())

200 {'probabilidad_churn': 0.67, 'riesgo': 'alto'}


### Ejercicio 1 — Endpoint `/health`

Agregá un endpoint `GET /health` que devuelva `{'status': 'ok'}` con código `200`. Es el endpoint que las plataformas de la Unidad 6 (Render, Railway) usan para chequear que tu servicio sigue respondiendo.

<details>
<summary>💡 Ver solución</summary>

```python
@app.get('/health')
def health():
    return {'status': 'ok'}

resp = cliente.get('/health')
print(resp.status_code, resp.json())
```

</details>

In [3]:
@app.get('/health')
def health():
    return {'status': 'ok'}

resp = cliente.get('/health')
print(resp.status_code, resp.json())

200 {'status': 'ok'}


## 3. Manejar errores explícitamente con `HTTPException`

Pydantic ya devuelve `422` automáticamente ante datos inválidos. Para otros errores de negocio (recurso no encontrado, reglas propias), usás `HTTPException`:

```python
from fastapi import HTTPException

METRICAS_DISPONIBLES = {'ventas', 'usuarios_activos', 'churn_promedio'}

@app.get('/metrics')
def metrics(nombre: str):
    if nombre not in METRICAS_DISPONIBLES:
        raise HTTPException(status_code=404, detail=f'Metrica {nombre} no encontrada')
    return {'nombre': nombre, 'valor': 12345}
```

In [4]:
from fastapi import HTTPException

METRICAS_DISPONIBLES = {'ventas', 'usuarios_activos', 'churn_promedio'}

@app.get('/metrics')
def metrics(nombre: str):
    if nombre not in METRICAS_DISPONIBLES:
        raise HTTPException(status_code=404, detail=f'Metrica {nombre} no encontrada')
    return {'nombre': nombre, 'valor': 12345}

### Ejercicio 2 — `GET /metrics` con filtros

Extendé el endpoint `/metrics` para que además reciba un query param opcional `periodo: str = 'mensual'`, valide que sea uno de `{'diario', 'semanal', 'mensual'}` (si no, `422` con `HTTPException`), y lo incluya en la respuesta.

<details>
<summary>💡 Ver solución</summary>

```python
PERIODOS_VALIDOS = {'diario', 'semanal', 'mensual'}

@app.get('/metrics')
def metrics(nombre: str, periodo: str = 'mensual'):
    if nombre not in METRICAS_DISPONIBLES:
        raise HTTPException(status_code=404, detail=f'Metrica {nombre} no encontrada')
    if periodo not in PERIODOS_VALIDOS:
        raise HTTPException(status_code=422, detail=f'Periodo {periodo} no valido')
    return {'nombre': nombre, 'periodo': periodo, 'valor': 12345}

resp = cliente.get('/metrics', params={'nombre': 'ventas', 'periodo': 'semanal'})
print(resp.status_code, resp.json())
```

</details>

In [5]:
PERIODOS_VALIDOS = {'diario', 'semanal', 'mensual'}

@app.get('/metrics')
def metrics(nombre: str, periodo: str = 'mensual'):
    if nombre not in METRICAS_DISPONIBLES:
        raise HTTPException(status_code=404, detail=f'Metrica {nombre} no encontrada')
    if periodo not in PERIODOS_VALIDOS:
        raise HTTPException(status_code=422, detail=f'Periodo {periodo} no valido')
    return {'nombre': nombre, 'periodo': periodo, 'valor': 12345}

resp = cliente.get('/metrics', params={'nombre': 'ventas', 'periodo': 'semanal'})
print(resp.status_code, resp.json())

200 {'nombre': 'ventas', 'valor': 12345}


## 4. CORS: permitir que una app web consuma tu API

Si tu API va a ser consumida desde un frontend en otro dominio (por ejemplo, una app en React en un dominio propio llamando a tu API en Render), el navegador bloquea la respuesta por CORS salvo que el servidor lo autorice explícitamente.

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=['https://miapp.com'],
    allow_methods=['GET', 'POST'],
    allow_headers=['*'],
)
```

Documentación oficial: [CORS en FastAPI](https://fastapi.tiangolo.com/tutorial/cors/)

In [7]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from starlette.middleware import Middleware

origins = ["https://miapp.com"]

middleware = [
    Middleware(
        CORSMiddleware,
        allow_origins=origins,
        allow_credentials=True,
        allow_methods=["GET", "POST"],
        allow_headers=["*"],
    )
]

app = FastAPI(middleware=middleware)


@app.get("/")
def read_root():
    return {"status": "ok"}

In [8]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

# 1. Crear instancia limpia
app = FastAPI()

# 2. Agregar middleware inmediatamente
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://miapp.com"],
    allow_credentials=True,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)


# 3. Definir rutas después
@app.get("/")
def read_root():
    return {"message": "listo"}

### Ejercicio 3 — Configurar CORS

¿Qué `allow_origins` configurarías si tu app web todavía no tiene dominio propio y estás probando localmente en `http://localhost:3000`? ¿Y si además querés que funcione desde cualquier origen mientras es un proyecto de curso?

<details>
<summary>💡 Ver solución</summary>

Para desarrollo local: `allow_origins=['http://localhost:3000']`. Para un proyecto de curso donde no importa restringir el origen: `allow_origins=['*']` — pero esto no se recomienda en producción real, porque cualquier sitio podría consumir tu API desde el navegador de un usuario.

</details>

## 5. De vuelta al resto del curso

- **Unidad 5:** el mismo patrón de `requests` que usaste para scrapear (Colab 1 de esa unidad) es el que usaría cualquier cliente — incluida tu propia app web — para consumir este microservicio.
- **Unidad 6:** este microservicio se dockeriza (mismo patrón `Dockerfile` con `uvicorn`) y se despliega en Render/Railway/HF Spaces igual que cualquier otra API FastAPI — el endpoint `/health` que armaste en el Ejercicio 1 es justamente lo que esas plataformas usan para confirmar que el deploy está sano.

## Mini-proyecto final: microservicio completo

Armá la versión final del microservicio con:

1. `POST /predict` — con el modelo Pydantic de entrada/salida y al menos una validación de negocio propia (por ejemplo, rechazar `antiguedad_meses` mayor a 600).
2. `GET /metrics` — con al menos dos filtros por query params.
3. `GET /health`.
4. CORS configurado.
5. `title`, `version` y `description` en la `app`, y `tags` en cada endpoint.
6. Al menos 4 pruebas con `TestClient` (casos exitosos y de error).

**Entregable:** el código completo de la API + las pruebas con `TestClient`. Como extensión opcional: dockerizala (Unidad 6) y desplegala en Render/Railway/HF Spaces.

In [9]:
!pip install -q fastapi uvicorn pydantic pytest httpx nest_asyncio

In [10]:
%%writefile main.py
from datetime import datetime
from typing import List, Optional
from fastapi import FastAPI, Query, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, field_validator

# --- Configuración de la App ---
app = FastAPI(
    title="Servicio de Scoring y Riesgo Crediticio",
    description="API para estimación de riesgo, cálculo de score y consulta de métricas operativas.",
    version="1.0.0",
)

# --- Configuración de CORS ---
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

# Base en memoria para registrar predicciones
HISTORICO_PREDICCIONES = []


# --- Modelos Pydantic ---
class PredictInput(BaseModel):
    ingresos: float = Field(..., gt=0, description="Ingresos netos mensuales")
    monto_solicitado: float = Field(..., gt=0, description="Monto del préstamo solicitado")
    antiguedad_meses: int = Field(..., ge=0, description="Antigüedad laboral en meses")
    segmento: str = Field(..., description="retail, pyme, corporativo")

    @field_validator("antiguedad_meses")
    @classmethod
    def validar_antiguedad(cls, valor: int) -> int:
        if valor > 600:
            raise ValueError("La antiguedad_meses no puede ser mayor a 600 meses (50 años).")
        return valor

    @field_validator("segmento")
    @classmethod
    def validar_segmento(cls, valor: str) -> str:
        segmentos_validos = {"retail", "pyme", "corporativo"}
        if valor.lower() not in segmentos_validos:
            raise ValueError(f"Segmento inválido. Valores aceptados: {segmentos_validos}")
        return valor.lower()


class PredictOutput(BaseModel):
    score: float
    decision: str
    tasa_riesgo: float
    timestamp: str


class MetricItem(BaseModel):
    segmento: str
    decision: str
    score: float
    monto_solicitado: float
    timestamp: str


# --- Endpoints ---
@app.get("/health", tags=["Monitoreo"], summary="Estado de salud de la API")
def get_health():
    return {
        "status": "healthy",
        "timestamp": datetime.utcnow().isoformat(),
        "version": app.version,
    }


@app.post("/predict", response_model=PredictOutput, tags=["Inferencia"], summary="Calcular scoring")
def predict(data: PredictInput):
    ratio_cuota_ingreso = data.monto_solicitado / (data.ingresos * 12)
    factor_antiguedad = min(data.antiguedad_meses / 120, 1.0)
    score_base = 750.0 + (factor_antiguedad * 100.0) - (ratio_cuota_ingreso * 200.0)
    score = round(max(300.0, min(950.0, score_base)), 2)

    if score >= 700:
        decision = "APROBADO"
        tasa_riesgo = 0.05
    elif score >= 550:
        decision = "EN_REVISION"
        tasa_riesgo = 0.18
    else:
        decision = "RECHAZADO"
        tasa_riesgo = 0.42

    resultado = {
        "score": score,
        "decision": decision,
        "tasa_riesgo": tasa_riesgo,
        "timestamp": datetime.utcnow().isoformat(),
    }

    HISTORICO_PREDICCIONES.append({
        "segmento": data.segmento,
        "decision": decision,
        "score": score,
        "monto_solicitado": data.monto_solicitado,
        "timestamp": resultado["timestamp"],
    })

    return resultado


@app.get("/metrics", response_model=List[MetricItem], tags=["Métricas"], summary="Consultar métricas con filtros")
def get_metrics(
    segmento: Optional[str] = Query(None, description="Filtro por segmento (retail, pyme, corporativo)"),
    decision: Optional[str] = Query(None, description="Filtro por resultado (APROBADO, EN_REVISION, RECHAZADO)"),
    score_min: Optional[float] = Query(None, ge=300, le=950, description="Filtro por score mínimo"),
):
    registros = HISTORICO_PREDICCIONES

    if segmento:
        registros = [r for r in registros if r["segmento"] == segmento.lower().strip()]
    if decision:
        registros = [r for r in registros if r["decision"] == decision.upper().strip()]
    if score_min is not None:
        registros = [r for r in registros if r["score"] >= score_min]

    return registros

Writing main.py


In [11]:
%%writefile test_main.py
import pytest
from fastapi import status
from fastapi.testclient import TestClient
from main import app, HISTORICO_PREDICCIONES

client = TestClient(app)

@pytest.fixture(autouse=True)
def limpiar_estado():
    HISTORICO_PREDICCIONES.clear()
    yield

def test_health_check_ok():
    response = client.get("/health")
    assert response.status_code == status.HTTP_200_OK
    assert response.json()["status"] == "healthy"

def test_predict_exitoso():
    payload = {
        "ingresos": 350000.0,
        "monto_solicitado": 500000.0,
        "antiguedad_meses": 48,
        "segmento": "retail",
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == status.HTTP_200_OK
    data = response.json()
    assert "score" in data
    assert data["decision"] in ["APROBADO", "EN_REVISION", "RECHAZADO"]

def test_predict_error_antiguedad_excedida():
    payload = {
        "ingresos": 200000.0,
        "monto_solicitado": 100000.0,
        "antiguedad_meses": 601,
        "segmento": "pyme",
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == status.HTTP_422_UNPROCESSABLE_ENTITY
    assert any("antiguedad_meses no puede ser mayor a 600" in err["msg"] for err in response.json()["detail"])

def test_predict_error_segmento_invalido():
    payload = {
        "ingresos": 150000.0,
        "monto_solicitado": 50000.0,
        "antiguedad_meses": 12,
        "segmento": "inmobiliario",
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == status.HTTP_422_UNPROCESSABLE_ENTITY
    assert any("Segmento inválido" in err["msg"] for err in response.json()["detail"])

def test_metrics_con_filtros():
    client.post("/predict", json={"ingresos": 500000.0, "monto_solicitado": 200000.0, "antiguedad_meses": 60, "segmento": "corporativo"})
    client.post("/predict", json={"ingresos": 80000.0, "monto_solicitado": 1500000.0, "antiguedad_meses": 6, "segmento": "retail"})

    res = client.get("/metrics?segmento=corporativo")
    assert res.status_code == status.HTTP_200_OK
    assert len(res.json()) == 1
    assert res.json()[0]["segmento"] == "corporativo"

Writing test_main.py


In [12]:
!pytest test_main.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.11.1, typeguard-4.6.0, anyio-4.14.2
collected 5 items                                                              

test_main.py::test_health_check_ok PASSED                                [ 20%]
test_main.py::test_predict_exitoso PASSED                                [ 40%]
test_main.py::test_predict_error_antiguedad_excedida PASSED              [ 60%]
test_main.py::test_predict_error_segmento_invalido PASSED                [ 80%]
test_main.py::test_metrics_con_filtros PASSED                            [100%]

=============================== warnings summary ===============================
test_main.py::test_predict_error_antiguedad_excedida
test_main.py::test_predict_error_segmento_invalido
  /usr/local/lib/python3.13/dist-packages/_pytest/python.py:157: Starlet

In [13]:
import subprocess
import threading
import time
import urllib.request
import uvicorn
from main import app


# 1. Iniciar uvicorn en segundo plano
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

# 2. Obtener la IP pública del runtime de Colab (se pide como password en localtunnel)
ip_colab = (
    urllib.request.urlopen("https://ipv4.icanhazip.com")
    .read()
    .decode("utf8")
    .strip()
)
print("=" * 60)
print(f"IP pública de Colab (Endpoint IP): {ip_colab}")
print("Pegá esa IP si la página de localtunnel te pide contraseña.")
print("=" * 60)

# 3. Lanzar túnel público
tunnel = subprocess.Popen(
    ["npx", "-y", "localtunnel", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# Esperar a que localtunnel imprima la URL
for line in tunnel.stdout:
    if "your url is:" in line.lower():
        url = line.split(":")[-1].strip()
        print(f"\nAPI lista en: https:{url}")
        print(f"Swagger Docs: https:{url}/docs")
        break

IP pública de Colab (Endpoint IP): 136.66.166.162
Pegá esa IP si la página de localtunnel te pide contraseña.

API lista en: https://petite-months-scream.loca.lt
Swagger Docs: https://petite-months-scream.loca.lt/docs


## Autoevaluación

- [ ] Mi API valida los datos de entrada con Pydantic
- [ ] Uso `response_model` para controlar qué se expone en las respuestas
- [ ] Manejo errores de negocio con `HTTPException` y códigos apropiados
- [ ] Tengo un endpoint `/health`
- [ ] Configuré CORS
- [ ] La documentación en `/docs` describe claramente cada endpoint

---

**Fin de la Unidad 4.** Con estos tres notebooks recorriste el ciclo completo: consumir APIs externas, construir las tuyas con FastAPI y Pydantic, y aplicarlo a un microservicio real de predicciones y métricas — listo para las Unidades 5 y 6.

In [14]:
from fastapi import status
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)


def test_autoevaluacion():
    print("=" * 65)
    print("EJECUTANDO AUTOEVALUACIÓN DE CRITERIOS DEL MICROSERVICIO")
    print("=" * 65)

    checklist = {
        "Valida datos con Pydantic": False,
        "Usa response_model en respuestas": False,
        "Manejo de errores / validaciones con códigos HTTP apropiados": False,
        "Endpoint /health implementado": False,
        "CORS configurado": False,
        "Documentación en /docs descriptiva": False,
    }

    # 1. Endpoint /health
    res_health = client.get("/health")
    if res_health.status_code == status.HTTP_200_OK and "status" in res_health.json():
        checklist["Endpoint /health implementado"] = True

    # 2. Validación de entrada Pydantic (debe rechazar payload corrupto o tipos inválidos)
    res_invalido = client.post(
        "/predict",
        json={"ingresos": "no-es-numero", "monto_solicitado": -100},
    )
    if res_invalido.status_code == status.HTTP_422_UNPROCESSABLE_ENTITY:
        checklist["Valida datos con Pydantic"] = True

    # 3. Manejo de errores de negocio (antigüedad > 600)
    res_negocio = client.post(
        "/predict",
        json={
            "ingresos": 100000,
            "monto_solicitado": 50000,
            "antiguedad_meses": 999,
            "segmento": "retail",
        },
    )
    if res_negocio.status_code in [
        status.HTTP_400_BAD_REQUEST,
        status.HTTP_422_UNPROCESSABLE_ENTITY,
    ]:
        checklist[
            "Manejo de errores / validaciones con códigos HTTP apropiados"
        ] = True

    # 4. response_model y documentación inspeccionando OpenAPI schema
    openapi_schema = app.openapi()

    # Verificar tags, título y descripción general
    tiene_info = bool(
        openapi_schema.get("info", {}).get("title")
        and openapi_schema.get("info", {}).get("description")
    )
    rutas_con_tags = all(
        "tags" in details
        for path in openapi_schema["paths"].values()
        for details in path.values()
    )
    if tiene_info and rutas_con_tags:
        checklist["Documentación en /docs descriptiva"] = True

    # Verificar que /predict y /metrics definan schemas de salida explícitos (response_model)
    paths = openapi_schema["paths"]
    resp_predict = paths.get("/predict", {}).get("post", {}).get("responses", {})
    resp_metrics = paths.get("/metrics", {}).get("get", {}).get("responses", {})

    schema_predict_ok = "200" in resp_predict and "content" in resp_predict["200"]
    schema_metrics_ok = "200" in resp_metrics and "content" in resp_metrics["200"]

    if schema_predict_ok and schema_metrics_ok:
        checklist["Usa response_model en respuestas"] = True

    # 5. Configuración de CORS
    cors_activo = any(
        m.cls.__name__ == "CORSMiddleware" for m in app.user_middleware
    )
    if cors_activo:
        checklist["CORS configurado"] = True

    # Imprimir reporte visual
    cumplidos = 0
    for criterio, aprobado in checklist.items():
        simbolo = "✅" if aprobado else "❌"
        print(f"{simbolo} {criterio}")
        if aprobado:
            cumplidos += 1

    print("-" * 65)
    print(f"Resultado final: {cumplidos}/{len(checklist)} criterios aprobados.")
    print("=" * 65)


test_autoevaluacion()

EJECUTANDO AUTOEVALUACIÓN DE CRITERIOS DEL MICROSERVICIO
✅ Valida datos con Pydantic
✅ Usa response_model en respuestas
✅ Manejo de errores / validaciones con códigos HTTP apropiados
✅ Endpoint /health implementado
✅ CORS configurado
✅ Documentación en /docs descriptiva
-----------------------------------------------------------------
Resultado final: 6/6 criterios aprobados.


/tmp/ipykernel_2681/577421323.py:100: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  test_autoevaluacion()
/tmp/ipykernel_2681/577421323.py:100: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  test_autoevaluacion()
